<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

`max.nn.Conv2d(permute=True)` keeps filters in OIHW order, which is how Paddle exported them. A stride-2 kernel-2 transposed convolution does not compile in this MAX build (`num_groups` fails to infer). With the kernel equal to the stride and no padding, the four taps do not overlap, so the transpose is four pointwise mixes interleaved on the spatial axes.

## Padding

In [4]:
#| echo: false
#| output: asis
show_doc(same_padding)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/conv.py#L15){target="_blank" style="float:right; font-size:smaller"}

### same_padding

```python
def same_padding(
    kernel:int
)->int:
```

Odd kernels, which is every kernel in these graphs, need `kernel // 2` on each side for a same-size convolution.

In [5]:
from fastcore.test import test_eq

test_eq(same_padding(3), 1)
test_eq(same_padding(9), 4)
same_padding(9)

4

In [8]:
#| echo: false
#| output: asis
show_doc(conv2d)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/conv.py#L28){target="_blank" style="float:right; font-size:smaller"}

### conv2d

```python
def conv2d(
    in_channels:int, out_channels:int, kernel_size:int | tuple[int, int], *, stride:int | tuple[int, int]=1,
    padding:int | tuple[int, int, int, int]=0, groups:int=1, bias:bool=True, device:DeviceRef=cpu:0
)->Conv2d:
```

`permute=True` is the whole layout contract. The checkpoint stores OIHW. MAX's default would be a different axis order, and [`load_checkpoint`](https://alix559.github.io/solid_lens/weights.html#load_checkpoint) compares shapes exactly.

## Stride-2 transpose

In [10]:
#| echo: false
#| output: asis
show_doc(ConvTranspose2x2)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/conv.py#L54){target="_blank" style="float:right; font-size:smaller"}

### ConvTranspose2x2

```python
def ConvTranspose2x2(
    in_channels:int, out_channels:int, *, bias:bool=True
)->None:
```

*Stride-2, kernel-2, pad-0 conv-transpose, written as interleaved matmuls.*

Each input pixel becomes a 2×2 block. The four taps are the slices of a weight laid out `(in, out, 2, 2)`, which is the ONNX conv-transpose layout. Width is interleaved first, then height, then the result is permuted back to NCHW.

In [13]:
#| echo: false
#| output: asis
show_doc(nchw_to_nhwc)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/conv.py#L103){target="_blank" style="float:right; font-size:smaller"}

### nchw_to_nhwc

```python
def nchw_to_nhwc(
    x:TensorValue
)->TensorValue:
```

In [15]:
#| echo: false
#| output: asis
show_doc(nhwc_to_nchw)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/conv.py#L108){target="_blank" style="float:right; font-size:smaller"}

### nhwc_to_nchw

```python
def nhwc_to_nchw(
    x:TensorValue
)->TensorValue:
```

Pooling in this MAX build wants NHWC. The backbone stays NCHW, so the pool sites convert on the way in and back out.